In [ ]:
import os
import re
import yaml
import gc
from pathlib import Path

import numpy as np
import pandas as pd

root = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "config.yml").exists())
with open(root / "config.yml") as f:
    DATA_PATH = yaml.safe_load(f)["data_path"]

# Preprocessing: 2024 and 2025 cohorts

Intervention and probability tables are harmonized for both cohorts. Contextual variables,
windows and baseline characteristics below are currently implemented for 2024.

## Variables Needed
* Contextual variables
* Intervention & Probability (see the harmonized schema below)
* BL Characteristics

### Intervention & Probability

Both cohorts are converted to one schema (`HARMONIZED_COLS`) and exported to
`processed/interventions_<cohort>.csv`.

| | 2024 | 2025 |
|---|---|---|
| Send | `action_id` 0/1, p_send = 0.5 | agent choice; `action_id` 0–9 kept as `action_code` |
| Domain | drawn per message from the remaining pool: `p_mood`, `p_steps`, `p_sleep` = P(domain \| send) | scheduled per decision (`decision_domain`, randomized weekly); P(domain \| send) = 1 for the scheduled domain |
| Category probabilities | conditional on send | unconditional (sum to p_send); divided by p_send here |
| Probability source | reconstructed pool sampling | `PROB_SOURCE_2025`: agent-marginal mixture (`p_marginal_*`, default) or selected agent (`p_*`) |
| Decision time | `hour_utc` = recorded attempt time, floored | `hour_utc` = schedule/cutoff label (often midnight), not the send time |

Harmonized probability columns (all conditional on sending except `p_send`, `p_no_send`):
`p_<category>` (e.g. `p_CSData`), `p_<domain>`, `p_strat_<code>` = P(strategy | send),
`p_<domain>_<code>` = P(domain, strategy | send). Blank probabilities are unknown, never zero.

#### Shared definitions and checks

In [ ]:
# ---- shared definitions and checks for intervention / probability tables

DOMAINS = ("mood", "steps", "sleep")
METRIC_LABELS = {"mood": "Mood", "steps": "Step Count", "sleep": "Hours of Sleep"}   # 2024 message_metric
STRATEGY_CODES = {"CS": "cogn", "BS": "behav", "MI": "motiv", "Mfn": "mindful", "DST": "distanc"}
CATEGORIES = ["CSNoData", "CSData", "BSNoData", "BSData", "MINoData", "MIData", "MfnData", "MfnNoData", "DSTNoData"]

# harmonized probability columns, all conditional on sending except p_send / p_no_send:
#   p_send, p_no_send        P(send), P(no send)
#   p_<category>             P(strategy, data version | send)
#   p_<domain>               P(message domain | send)
#   p_strat_<code>           P(strategy | send) = p_<code>Data + p_<code>NoData
#   p_<domain>_<code>        P(domain, strategy | send)
P_STRAT = [f"p_strat_{c}" for c in STRATEGY_CODES]
P_DOMAIN = [f"p_{d}" for d in DOMAINS]
P_JOINT = [f"p_{d}_{c}" for d in DOMAINS for c in STRATEGY_CODES]
P_CAT = [f"p_{c}" for c in CATEGORIES]

HARMONIZED_COLS = [
    "decision_id", "participant_id", "decision_date_utc", "decision_time_utc",
    "action_id",            # 0 = no send, 1 = send (both cohorts)
    "message_metric",       # domain of the sent message (2024 labels); NaN for no-send
    "scheduled_domain",     # 2025: domain scheduled for the decision (also for no-sends); NaN in 2024
    "strategy", "strategy_code", "personalization",
    "p_send", "p_no_send", *P_CAT, *P_DOMAIN, *P_STRAT, *P_JOINT,
]


def strategy_code(label):
    """Strategy label -> code: exact code (case-insensitive) or a unique pattern match."""
    low = str(label).lower()
    exact = [c for c in STRATEGY_CODES if c.lower() == low]
    if exact:
        return exact[0]
    hits = [c for c, pat in STRATEGY_CODES.items() if re.search(pat, low)]
    return hits[0] if len(hits) == 1 else None


def add_strategy_code(df: pd.DataFrame) -> pd.DataFrame:
    code_map = {s: strategy_code(s) for s in df["strategy"].dropna().unique()}
    print("strategy codes:", code_map)
    assert None not in code_map.values(), f"unmatched strategy labels: {[s for s, c in code_map.items() if c is None]}"
    assert len(set(code_map.values())) == len(code_map), "two strategy labels map to the same code"
    return df.assign(strategy_code=df["strategy"].map(code_map))


def add_strategy_probs(df: pd.DataFrame) -> pd.DataFrame:
    """p_strat_<code> = p_<code>Data + p_<code>NoData (DST has only NoData)."""
    df = df.copy()
    for code in STRATEGY_CODES:
        cols = [c for c in (f"p_{code}Data", f"p_{code}NoData") if c in df.columns]
        assert cols, f"no probability columns for strategy {code}"
        df[f"p_strat_{code}"] = df[cols].sum(axis=1, min_count=len(cols))
    return df


def check_harmonized(df: pd.DataFrame, strict_positivity: bool = True) -> None:
    """Checks shared by both cohorts on the harmonized table."""
    assert df["decision_id"].is_unique, "duplicate decision_id"
    assert not df.duplicated(["participant_id", "decision_date_utc"]).any(), "more than one decision per participant-date"
    assert df["action_id"].isin([0, 1]).all()
    assert df[["p_send", "p_no_send"]].notnull().all().all(), "missing send probability"
    assert np.isclose(df["p_send"] + df["p_no_send"], 1, atol=1e-4).all()

    sent = df["action_id"] == 1
    # message attributes present <=> sent
    for c in ("message_metric", "strategy_code"):
        assert df.loc[sent, c].notnull().all(), f"{c} missing for a send"
        assert df.loc[~sent, c].isnull().all(), f"{c} present for a no-send"

    # conditional-on-send probabilities: defined, in [0, 1] and summing correctly when p_send > 0
    pos = df["p_send"] > 0
    probs = [*P_CAT, *P_DOMAIN, *P_STRAT, *P_JOINT]
    sub = df.loc[pos, probs]
    assert sub.notnull().all().all(), "missing conditional probability where p_send > 0"
    assert ((sub >= -1e-9) & (sub <= 1 + 1e-9)).all().all()
    for cols in (P_CAT, P_DOMAIN, P_STRAT, P_JOINT):
        assert np.isclose(sub[cols].sum(axis=1), 1, atol=1e-4).all(), f"{cols[0]}... do not sum to 1"
    for d in DOMAINS:
        assert np.isclose(sub[[f"p_{d}_{c}" for c in STRATEGY_CODES]].sum(axis=1), sub[f"p_{d}"], atol=1e-4).all()
    for c in STRATEGY_CODES:
        assert np.isclose(sub[[f"p_{d}_{c}" for d in DOMAINS]].sum(axis=1), sub[f"p_strat_{c}"], atol=1e-4).all()

    assert ((df[probs] >= 0) | (df[probs].isna())).all().all()

    # positivity of the observed action
    s = df.loc[sent]
    dom = s["message_metric"].map({v: k for k, v in METRIC_LABELS.items()})
    p_obs = s["p_send"] * np.array([s.at[i, f"p_{d}_{c}"] for i, d, c in zip(s.index, dom, s["strategy_code"])])
    n_zero_send = int((p_obs <= 0).sum())
    n_zero_nosend = int((df.loc[~sent, "p_no_send"] <= 0).sum())
    print(f"rows: {len(df)}; sends: {sent.sum()}; observed action with probability 0: "
          f"{n_zero_send} sends, {n_zero_nosend} no-sends; p_send in {{0, 1}}: {(~df['p_send'].between(0, 1, inclusive='neither')).sum()}")
    if strict_positivity:
        assert n_zero_send == 0 and n_zero_nosend == 0, "observed action had probability 0"


def export_interventions(df: pd.DataFrame, cohort: str, extra: list[str] = ()) -> None:
    out = df[[*HARMONIZED_COLS, *extra]].set_index("decision_id")
    out.to_csv(os.path.join(DATA_PATH, f"processed/interventions_{cohort}.csv"))
    print(f"exported processed/interventions_{cohort}.csv: {len(out)} decisions, {out['participant_id'].nunique()} participants")

#### 2024

In [ ]:
def harmonize_2024(raw: pd.DataFrame, start: str = "2024-07-01") -> pd.DataFrame:
    """2024 behavior-agent table -> harmonized schema.

    2024: binary send/no-send (p_send = 0.5); message domain and strategy drawn from the
    remaining message pool, so p_<category>, p_<domain> and p_<domain>_<code> are already
    conditional on sending. Pre-internship records are dropped.
    """
    intv = raw.copy()
    p_cols = intv.columns[intv.columns.str.startswith("p_")]

    # convert to datetime
    intv["recorded_time_utc"] = pd.to_datetime(intv["recorded_time_utc"], utc=True)
    intv["hour_utc"] = pd.to_datetime(intv["hour_utc"], utc=True)
    intv["decision_date"] = pd.to_datetime(intv["decision_date"])

    # verify decision_date reflects time in UTC
    assert (intv["decision_date"] == intv["hour_utc"].dt.date).all()

    # verify Preinternship notification => before study start
    notif = intv["notification_id"].str.extract(r"(\D+)\d*(?:-\d+-\d)?$")[0]
    p = notif == "Preinternship"
    q = intv["recorded_time_utc"] < pd.Timestamp(start, tz="UTC")
    assert (~p | q).all()
    # verify after study start => not Preinternship notification
    assert (q | ~p).all()

    # verify NoMessage notification <=> action_id == 0
    p = notif == "NoMessage"
    q = intv["action_id"] == 0
    assert (~p | q).all()
    assert (~q | p).all()

    # filter out preinternship
    intv = intv.loc[intv["recorded_time_utc"] >= pd.Timestamp(start, tz="UTC")]

    # verify action_id == 1 <=> core_message_id
    assert (~(intv["action_id"] == 1) | (~intv["core_message_id"].isna())).all()
    assert (intv["core_message_id"].isna() | (intv["action_id"] == 1)).all()

    # verify probability is not missing
    assert intv[p_cols].notnull().all().all()

    # verify no send <=> missing metric & missing strategy
    p = intv["action_id"] == 0
    q = intv["message_metric"].isnull() & intv["strategy"].isnull()
    assert (~p | q).all()
    assert (~q | p).all()

    # verify chosen message has consistent domain and strategy
    assert intv.groupby("core_message_id")[["message_metric", "strategy"]].nunique().eq(1).all(axis=1).all()

    # verify p_send = p_no_send = 0.5 and category probabilities (conditional on send) sum to 1
    assert np.isclose(intv[["p_send", "p_no_send"]], 0.5).all().all()
    assert np.isclose(intv[P_CAT].sum(axis=1), 1).all()

    intv = add_strategy_probs(intv)
    intv = add_strategy_code(intv)
    if "personalization" not in intv.columns:
        intv["personalization"] = np.nan

    return intv.rename(columns={"hour_utc": "decision_time_utc", "decision_date": "decision_date_utc"})\
               .assign(scheduled_domain=np.nan)

In [ ]:
PROB_FILES = {"2024": "2024BehaviorAgentProbabilities.csv", "2025": "2025BehaviorAgentProbabilities.csv"}

raw = pd.read_csv(os.path.join(DATA_PATH, PROB_FILES["2024"]))
interventions = harmonize_2024(raw)
check_harmonized(interventions)
export_interventions(interventions, "2024")

del raw, interventions
gc.collect()

### Contextual Variables & Outcomes

In [ ]:
def wide2long(df, regex=r'^(Fitbit|Garmin|Healthkit)', extra=('Mood',), pid='participant_id'):
    """Device columns -> long: one row per index entry and device, one column per stream.

    Keeps all of `df`'s index levels and adds `device_type`. Columns in
    `extra` (e.g. Mood) are not device-specific: they are copied onto every
    device the participant (`pid` index level) has data from, including
    rows where that device recorded nothing.
    """
    keys = list(df.index.names)

    dev = df.filter(regex=regex)
    dev.columns = pd.MultiIndex.from_tuples(
        [c.split('_', 1) for c in dev.columns], names=['device_type', 'outcome']
    )
    long = dev.stack('device_type').dropna(how='all').reset_index()

    extra = [c for c in extra if c in df.columns]
    if extra:
        devices = long[[pid, 'device_type']].drop_duplicates()   # each participant's devices
        ext = df[extra].dropna(how='all').reset_index().merge(devices, on=pid)
        long = long.merge(ext, on=[*keys, 'device_type'], how='outer')

    return long.set_index([*keys, 'device_type']).sort_index()


def naive_local(s: pd.Series) -> pd.Series:
    """Local wall-clock time without offset. Accepts ISO strings ('YYYY-MM-DD HH:MM:SS...' or
    'YYYY-MM-DDTHH:MM:SS...'); any fractional seconds or trailing UTC offset is ignored."""
    s = s.astype(str).str.replace("T", " ", n=1, regex=False).str.slice(0, 19)
    return pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")


fnames_dfs = {
    '2024': 'dfHourly2024.csv',
    # '2025': 'dfHourly2025.csv'
}
dfs = {k: pd.read_csv(os.path.join(DATA_PATH, v)) for k, v in fnames_dfs.items()}

contexts = dfs['2024'].copy()
contexts = contexts.rename(columns={
    'time_utc': 'context_time_utc',
    'time_local': 'context_time_local'
})
# parse UTC times; rows without a UTC time are dropped (UTC is not inferred from local time)
contexts['context_time_utc'] = pd.to_datetime(contexts['context_time_utc'], utc=True)
n_miss = contexts['context_time_utc'].isna().sum()
print(f"dropping {n_miss} rows ({n_miss / len(contexts):.1%}) with missing UTC time")
contexts = contexts.loc[contexts['context_time_utc'].notna()]
# keep from 24 h before the first possible decision
contexts = contexts.loc[contexts["context_time_utc"] >= pd.Timestamp("2024-06-30", tz="UTC")]
# replace 0 mood with NaN
contexts['Mood'] = contexts['Mood'].replace(0, np.nan)

# convert from wide to long
idx = ['participant_id', 'context_time_utc', 'context_time_local']
contexts_wide = contexts.set_index(idx)
contexts_long = wide2long(contexts_wide)
del contexts, contexts_wide
gc.collect()

# combined Healthkit: watch, NaNs filled with phone.
# <stream>_from_phone = 1 if the hour's value came from the phone, 0 if from the watch,
# NaN if neither has data (device streams only; Mood is not device-specific)
watch = contexts_long.xs('HealthkitWatch', level='device_type')
phone = contexts_long.xs('HealthkitPhone', level='device_type')
watch, phone = watch.align(phone)
dev_cols = [c for c in watch.columns if c != 'Mood']
from_phone = (
    (watch[dev_cols].isna() & phone[dev_cols].notna()).astype(float)
    .where(watch[dev_cols].notna() | phone[dev_cols].notna())
)
combined = (
    pd.concat({'Healthkit': watch.combine_first(phone).join(from_phone.add_suffix('_from_phone'))},
              names=['device_type'])
    .reorder_levels([*idx, 'device_type'])
)
contexts_long = pd.concat([contexts_long, combined])

# export
contexts_long.to_csv(os.path.join(DATA_PATH, "processed/contexts_2024.csv"))

del fnames_dfs, dfs, watch, phone, from_phone, combined, contexts_long
gc.collect()

In [ ]:
# column suffix -> aggregation (first match wins; default is mean)
RULES = {
    "steps": "sum",           # total
    "sleep_minutes": "sum",   # total minutes asleep
    "hr_avg": "mean",
    "hr_max": "max",
    "RHR": "median",          # usually a daily value repeated across hours
    "Mood": "mean",
    "_from_phone": "mean",    # share of hours with data filled from the phone (combined Healthkit)
}
 
# columns that also get a within-window SD
SD_SUFFIXES = ("hr_avg", "Mood")
 
 
def _rule(col: str) -> str:
    return next((how for suffix, how in RULES.items() if col.endswith(suffix)), "mean")


def aggregate(df: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    """Aggregate hourly rows to one row per group, one rule per data stream.

    Adds `<col>_sd` = SD of hourly values for each hr_avg and Mood column
    (NaN if fewer than 2 values), and `<col>_n` = number of non-null hours
    per group for each stream (not for `_from_phone` columns). Groups with no data
    for a stream are NaN (not 0, which plain sum gives). Coverage thresholds are
    applied in the analysis.
    """
    keys = list(keys)
    g = df.groupby(keys)
    n = g.count()
    cols = [c for c in df.columns if c not in keys]  # keys may be columns or index levels
    out = g.agg({c: _rule(c) for c in cols}).where(n > 0)

    # within-window variability
    sd_cols = [c for c in cols if c.endswith(SD_SUFFIXES)]
    sd = g[sd_cols].std().add_suffix("_sd")
 
    counts = n[[c for c in n.columns if not c.endswith("_from_phone")]]
    return out.join(sd).join(counts.add_suffix("_n"))
 
 
def hour_utc(s: pd.Series) -> pd.Series:
    """Parse to UTC timestamps and floor to the start of the hour."""
    return pd.to_datetime(s, utc=True).dt.floor("h")


def build_window_grid(
    interventions: pd.DataFrame,
    period_h: int = 24,
    pid: str = "participant_id",
    dec_time: str = "decision_time_utc",
) -> pd.DataFrame:
    """One row per participant x window index t, whether or not a decision occurred.

    Each participant's first decision defines t = 0. Each later decision advances t
    by its gap from the previous decision in units of `period_h`, rounded and at
    least 1 (a gap of ~1 day -> +1, ~2 days -> +2), so drifting or shifted decision
    times never collide and skipped days get their own index. Window t starts at the
    decision time when there is one; otherwise it is interpolated from the nearest decision in steps of
    `period_h`. Window t ends where window t+1 starts (the last ends period_h later),
    so windows are contiguous and every hour belongs to exactly one window.
    t = -1 is the window before the first decision, used only as history.
    """
    intv = (interventions.assign(**{dec_time: hour_utc(interventions[dec_time])})
                         .sort_values([pid, dec_time]))
    step = pd.Timedelta(hours=period_h)
    # index from consecutive gaps: each decision advances t by its rounded gap
    # (at least 1), so drift in decision times never makes two decisions collide
    gap = intv.groupby(pid)[dec_time].diff() / step
    inc = gap.round().clip(lower=1).fillna(0).astype(int)
    intv["t"] = inc.groupby(intv[pid]).cumsum()
    assert not intv.duplicated([pid, "t"]).any(), "duplicate window index"

    tmax = intv.groupby(pid)["t"].max()
    grid = pd.DataFrame({
        pid: np.repeat(tmax.index.values, tmax.values + 2),
        "t": np.concatenate([np.arange(-1, m + 1) for m in tmax.values]),
    }).merge(intv[[pid, "t", dec_time]], on=[pid, "t"], how="left")

    # interpolate start times for windows without a decision
    known_t = grid["t"].where(grid[dec_time].notna())
    g = grid.groupby(pid)
    f_time, f_t = g[dec_time].ffill(), known_t.groupby(grid[pid]).ffill()
    b_time, b_t = g[dec_time].bfill(), known_t.groupby(grid[pid]).bfill()
    grid["start"] = (f_time + (grid["t"] - f_t) * step).fillna(b_time - (b_t - grid["t"]) * step)

    grid["end"] = grid.groupby(pid)["start"].shift(-1).fillna(grid["start"] + step)
    assert (grid["end"] > grid["start"]).all(), "window start times are not increasing"
    grid["span_h"] = (grid["end"] - grid["start"]) / pd.Timedelta("1h")
    grid["has_decision"] = grid[dec_time].notna()
    return grid


def windows_by_index(
    contexts: pd.DataFrame,
    interventions: pd.DataFrame,
    streams: list[str],
    period_h: int = 24,
    by: list[str] = ("participant_id", "device_type"),
    ctx_time: str = "context_time_utc",
    dec_time: str = "decision_time_utc",
) -> pd.DataFrame:
    """Aggregate hourly contexts into the window grid.

    Returns one row per `by` + t for t >= 0, with
      <feature>_post : window t       (proximal outcome when a decision starts window t)
      <feature>_pre  : window t - 1   (history / moderators)
    plus start, end, span_h, has_decision and the intervention columns
    (missing when there is no decision in window t). Rows exist for every window
    of every device a participant has data from, even windows with no data.
    """
    by = list(by)
    pid = by[0]
    ts = "datetime64[ns, UTC]"
    grid = build_window_grid(interventions, period_h, pid, dec_time)
    grid[["start", "end"]] = grid[["start", "end"]].astype(ts)

    ctx = contexts.assign(**{ctx_time: hour_utc(contexts[ctx_time]).astype(ts)})
    bad = ctx[[pid, ctx_time]].isna().any(axis=1)
    if bad.any():
        print(f"dropping {bad.sum()} context rows with missing {pid} or {ctx_time}")
    ctx = ctx.loc[~bad].sort_values(ctx_time)
    # assign each hour to the window with start <= c, then drop hours past its end
    hourly = pd.merge_asof(
        ctx[[*by, ctx_time, *streams]], grid[[pid, "t", "start", "end"]].sort_values("start"),
        left_on=ctx_time, right_on="start", by=pid, direction="backward",
    )
    hourly = hourly.loc[hourly["t"].notna() & (hourly[ctx_time] < hourly["end"])]
    hourly["t"] = hourly["t"].astype(int)

    # TODO: verify interpretation of sleep = 0 & handle potential edge cases
    feats = aggregate(hourly[[*by, "t", *streams]], keys=[*by, "t"]).reset_index()

    # full grid for every device a participant has data from
    devices = ctx[by].drop_duplicates()
    rows = grid.merge(devices, on=pid)
    post = feats.rename(columns={c: f"{c}_post" for c in feats.columns if c not in [*by, "t"]})
    pre = (feats.assign(t=feats["t"] + 1)
                .rename(columns={c: f"{c}_pre" for c in feats.columns if c not in [*by, "t"]}))
    span_pre = grid[[pid, "t", "span_h"]].assign(t=grid["t"] + 1).rename(columns={"span_h": "span_h_pre"})

    out = (rows.rename(columns={"span_h": "span_h_post"})
               .merge(post, on=[*by, "t"], how="left")
               .merge(pre, on=[*by, "t"], how="left")
               .merge(span_pre, on=[pid, "t"], how="left"))
    out = out.loc[out["t"] >= 0]

    intv = interventions.assign(**{dec_time: hour_utc(interventions[dec_time])})
    out = out.merge(intv, on=[pid, dec_time], how="left")
    return out.set_index([*by, "t"]).sort_index()
 

In [ ]:
# ---- derived variables (fixed definitions; thresholds are applied in the analysis)

def add_derived_variables(feats: pd.DataFrame) -> pd.DataFrame:
    """Fixed variable definitions shared by all analyses (no thresholds).

      log_steps_{pre,post} : log(1 + steps)
      <stream>_cov_{w}     : <stream>_n_{w} / span_h_{w}, share of the window's hours
                             with data for that stream (thresholds are applied in the analysis)
    """
    out = feats.copy()
    for w in ("pre", "post"):
        out[f"log_steps_{w}"] = np.log1p(out[f"steps_{w}"])
        span = out[f"span_h_{w}"]
        for n_col in [c for c in out.columns if c.endswith(f"_n_{w}")]:
            stream = n_col[: -len(f"_n_{w}")]
            out[f"{stream}_cov_{w}"] = out[n_col].fillna(0) / span
    return out


def add_local_time(
    feats: pd.DataFrame,
    contexts: pd.DataFrame,
    pid: str = "participant_id",
    ctx_utc: str = "context_time_utc",
    ctx_local: str = "context_time_local",
    tolerance_h: int = 12,
) -> pd.DataFrame:
    """Local start time of each window, from the participant's UTC offset at the nearest
    context hour (within `tolerance_h`). For decision rows, window start = decision time.
    Uses `naive_local` (defined with the contextual variables).

      start_local : local wall-clock time of the window start
      dow         : day of week of start_local, 1 = Monday ... 7 = Sunday
      weekend     : 1 if Saturday or Sunday
    """
    ts = "datetime64[ns, UTC]"
    off = contexts[[pid, ctx_utc, ctx_local]].drop_duplicates([pid, ctx_utc])
    off = off.assign(**{ctx_utc: hour_utc(off[ctx_utc]).astype(ts)})
    off["utc_offset"] = naive_local(off[ctx_local]) - off[ctx_utc].dt.tz_localize(None)
    off = off.dropna(subset=[ctx_utc, "utc_offset"]).sort_values(ctx_utc)

    f = feats.reset_index()
    f["_row"] = np.arange(len(f))
    left = f[[pid, "start", "_row"]].assign(start=lambda x: x["start"].astype(ts)).sort_values("start")
    m = pd.merge_asof(left, off[[pid, ctx_utc, "utc_offset"]], left_on="start", right_on=ctx_utc,
                      by=pid, direction="nearest", tolerance=pd.Timedelta(hours=tolerance_h))
    m = m.set_index("_row").reindex(f["_row"])

    f["start_local"] = f["start"].astype(ts).dt.tz_localize(None) + m["utc_offset"].values
    f["dow"] = f["start_local"].dt.dayofweek + 1
    f["weekend"] = (f["dow"] >= 6).astype("Int64").where(f["dow"].notna())
    miss = f["start_local"].isna().mean()
    print(f"local time missing for {miss:.1%} of windows")
    if miss > 0.5:
        print("check the format of", ctx_local, "- examples:", contexts[ctx_local].dropna().astype(str).head(3).tolist())
    return f.drop(columns="_row").set_index(feats.index.names)

### BL Characteristics

Used (`bl_cols`):
* age: `Age`
* sex: `Sex` (0 = male, 1 = female)
* depression history: `depr0`
* currently on medication: `medication0` (derived from `medication0_None`)
* depressive symptoms: `PHQtot0`
* anxiety symptoms: `GADtot0`
* neuroticism: `Neu0_published`
* stressful life events: `SLE0`

Not yet included:
* gender
* race/ethnicity: `Ethnicity_[1-7,9]`
* marital: `Marital`
* relatives with depression: `deprRelat`; age at first depression: `deprAge`
* ever on treatment: `deprTreat0`; treatment type: `specTreat0_*`
* medication type: `medication0_*`; current therapy: `therapy0`
* specialty: `SpecialtyAAMC`

In [ ]:
interventions = pd.read_csv(os.path.join(DATA_PATH, "processed/interventions_2024.csv"))
contexts_long = pd.read_csv(os.path.join(DATA_PATH, "processed/contexts_2024.csv"))

streams = [c for c in contexts_long.columns if c not in ('participant_id', 'context_time_utc', 'context_time_local', 'device_type')]

feats = windows_by_index(contexts_long, interventions, streams=streams)

# filter out data before 7/1
feats = feats.loc[(feats['start'] >= pd.Timestamp("2024-07-01", tz="UTC"))]

# fixed derived variables: log steps, coverage shares, local day of week
feats = add_derived_variables(feats)
feats = add_local_time(feats, contexts_long)

# read
data = pd.read_csv(os.path.join(DATA_PATH, '2024SurveyData.csv'))
# currently on medication
data['medication0'] = data[data.columns[data.columns.str.match(r"^medication\d_None$")]]\
    .all(axis=1).astype(int) * -1 + 1
# remap sex
data['Sex'] = data['Sex'].map({1: 0, 2: 1})
# subset
bl_cols = ['Age', 'Sex', 'depr0', 'medication0', 'PHQtot0', 'GADtot0', 'Neu0_published', 'SLE0']
bl = data.set_index("PARTICIPANTIDENTIFIER")[bl_cols].rename_axis("participant_id")
# add BL characteristics
feats = feats.reset_index().merge(bl.reset_index(), how='left', on='participant_id')\
    .set_index(["participant_id", "device_type"])
n_no_bl = feats.loc[feats[bl_cols].isna().all(axis=1)].index.get_level_values("participant_id").nunique()
print(f"participants without baseline survey data: {n_no_bl}")

# TODO: subset feats to necessary variables

# export
feats.to_csv(os.path.join(DATA_PATH, "processed/data_final_2024.csv"))

del feats, data, bl
gc.collect()